In [1]:
from myosuite.utils import gym
from tqdm import tqdm
from stable_baselines3 import PPO
import numpy as np
import imageio
from IPython.display import Video, display
import os
import pickle

MyoSuite:> Registering Myo Envs


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


### Create an environment and visualize a random policy

In [2]:
realtime_visualization = False

env = gym.make('myoChallengeOslRunFixed-v0')
print('List of cameras available', [env.unwrapped.mj_model.camera(i).name for i in range(env.unwrapped.mj_model.ncam)])
env.reset()
frames = []

for _ in tqdm(range(200)):

    if realtime_visualization:
        env.unwrapped.mj_render()
        action = env.action_space.sample() # take a random action
        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            env.reset()

    else:
        frame = env.unwrapped.mj_renderer.render_offscreen(camera_id=1)
        frames.append(frame)
        env.step(env.action_space.sample()) # take a random action
        env.close()

os.makedirs('videos', exist_ok=True) # make a local copy
try:
    imageio.mimwrite(f"videos/temp.mp4", frames, fps=1.0 / env.unwrapped.dt)
except Exception as e:
    print('WARNING: imageio.mimwrite failed:', e)

# show in the notebook
display(Video(f"videos/temp.mp4", embed=True))

    MyoSuite: A contact-rich simulation suite for musculoskeletal motor control
        Vittorio Caggiano, Huawei Wang, Guillaume Durandau, Massimo Sartori, Vikash Kumar
        L4DC-2019 | https://sites.google.com/view/myosuite
    


c:\Users\ST000082\Documents\Codes\myosuite\.venv\Lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\ST000082\Documents\Codes\myosuite\.venv\Lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
c:\Users\ST000082\Documents\Codes\myosuite\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


List of cameras available ['topview', 'side_view', 'agent_view']


  0%|          | 0/200 [00:00<?, ?it/s]c:\Users\ST000082\Documents\Codes\myosuite\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
100%|██████████| 200/200 [00:17<00:00, 11.19it/s]


In [3]:
print("Observation space:", env.observation_space)
print("Action space:", env.action_space) # 54 muscles
print("Obs keys:", env.unwrapped.obs_dict.keys()) # observation of the prosthesis is only at the socket interface

Observation space: Box(-10.0, 10.0, (260,), float32)
Action space: Box(-1.0, 1.0, (54,), float32)
Obs keys: dict_keys(['time', 'terrain', 'internal_qpos', 'internal_qvel', 'grf', 'socket_force', 'torso_angle', 'muscle_length', 'muscle_velocity', 'muscle_force', 'act', 'model_root_pos', 'model_root_vel', 'hfield'])


### Train a policy

In [4]:
env = gym.make('myoChallengeOslRunFixed-v0')
model = PPO("MlpPolicy", env, verbose=0)
model.learn(total_timesteps=100)



c:\Users\ST000082\Documents\Codes\myosuite\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
c:\Users\ST000082\Documents\Codes\myosuite\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


In [5]:
# evaluate policy
all_rewards = []
for _ in tqdm(range(20)): # 20 random targets
  ep_rewards = []
  done = False
  obs, _ = env.reset()
  for _ in range(40):
      # get the next action from the policy
      action, _ = model.predict(obs)
      # take an action based on the current observation
      obs, reward, done, _, info = env.step(action)
      ep_rewards.append(reward)
  all_rewards.append(np.sum(ep_rewards))
print(f"Average reward: {np.mean(all_rewards)} over 20 episodes")

100%|██████████| 20/20 [00:01<00:00, 10.54it/s]

Average reward: 26.015192758543897 over 20 episodes


### Load a policy

In [ ]:
pth = '../myosuite/agents/baslines_NPG/'
policy = os.path.join(
    pth,
    "C:\\Users\\ST000082\\Documents\\Codes\\myosuite\\myosuite\\agents\\baslines_NPG\\myoElbowPose1D6MExoRandom-v0\\2022-02-26_21-16-27\\36_env=myoElbowPose1D6MExoRandom-v0,seed=1\\iterations\\best_policy.pickle",
)

if not os.path.exists(policy):
    print(f"Policy file not found at {policy}, skipping policy loading in this environment.")
    pi = None
else:
    with open(policy, 'rb') as f:
        pi = pickle.load(f)

In [ ]:
from myosuite.utils import gym
# Include the locomotion track environment, uncomment to select the manipulation challenge
env = gym.make('myoChallengeOslRunRandom-v0')
env = env.unwrapped  # tous les appels bas-niveau passent par sim_env

# env = gym.make('myoChallengeBimanual-v0')


env.reset()

# Repeat 1000 time steps
for _ in range(1000):

    # Activate mujoco rendering window
    env.unwrapped.mj_render()

    # Select skin group
    geom_1_indices = np.where(env.mj_model.geom_group == 1)
    # Change the alpha value to make it transparent
    env.mj_model.geom_rgba[geom_1_indices, 3] = 0


    # Get observation from the envrionment, details are described in the above docs
    obs = env.get_obs()
    # current_time = obs['time']
    #print(current_time)


    # Take random actions
    action = env.action_space.sample()


    # Environment provides feedback on action
    next_obs, reward, terminated, truncated, info = env.step(action)


    # Reset training if env is terminated
    if terminated:
        next_obs, info = env.reset()

Exception ignored in: <function Renderer.__del__ at 0x000001E7323EC220>
Traceback (most recent call last):
  File "c:\Users\ST000082\Documents\Codes\myosuite\myosuite\renderer\renderer.py", line 145, in __del__
    self.close()
  File "c:\Users\ST000082\Documents\Codes\myosuite\myosuite\renderer\mj_renderer.py", line 162, in close
    quit()
    ^^^^
NameError: name 'quit' is not defined


Exception ignored in: <function Renderer.__del__ at 0x000001E7323EC220>
Traceback (most recent call last):
  File "c:\Users\ST000082\Documents\Codes\myosuite\myosuite\renderer\renderer.py", line 145, in __del__
    self.close()
  File "c:\Users\ST000082\Documents\Codes\myosuite\myosuite\renderer\mj_renderer.py", line 162, in close
    quit()
  File "<frozen _sitebuiltins>", line 26, in __call__
SystemExit: None
